# 23-24 · Przeczytaj git diff i zmiany grup

Praktyka do sekcji [„Samosprawdzanie zmian i dyscyplina commitów”](/pl/chapters/rozdzial-23/23-28-git-kommit.html).

## Reproducible local environment

```bash
git clone https://github.com/Cartesian-School/safesort.git
cd safesort
python3.14 -m venv .venv
source .venv/bin/activate
# Windows PowerShell: .venv\Scripts\Activate.ps1
python -m pip install -U pip
python -m pip install -e ".[dev]"
python -m pip install jupyter ipykernel
python -m ipykernel install --user --name safesort-py314 --display-name "SafeSort Python 3.14"
jupyter lab
```

Select the **SafeSort Python 3.14** kernel. The diagnostic cell below must
point into this `.venv` and the cloned `src/safesort` tree.

In [ ]:
import sys
import safesort

print(sys.executable)
print(safesort.__file__)

## Cel

`git diff` nie jest wykonywany, lecz odczytywany. To nie ćwiczenie o uruchomieniu kodu, lecz o ustaleniu, co dokładnie się zmieniło i sformułowaniu krótkiej, szczerej wiadomości commit-owej o tej zmianie.

## Example — wynik git diff przed zatwierdzeniem

In [ ]:
PRIMER_DIFF = """diff --git a/src/safesort/duplicates.py b/src/safesort/duplicates.py
index 1a2b3c4..5d6e7f8 100644
--- a/src/safesort/duplicates.py
+++ b/src/safesort/duplicates.py
@@ -40,6 +40,9 @@ def find_duplicates(files, chunk_size=DEFAULT_CHUNK_SIZE):
     for size, candidates in by_size.items():
         if len(candidates) < 2:
             continue
+
+        if size == 0:
+            logger.info("Пустые файлы тоже считаются дубликатами: %d штук", len(candidates))
         by_digest = defaultdict(list)
diff --git a/tests/test_duplicates.py b/tests/test_duplicates.py
index 9f8e7d6..2c3b4a5 100644
--- a/tests/test_duplicates.py
+++ b/tests/test_duplicates.py
@@ -12,3 +12,10 @@ def test_identical_content_files_are_grouped(tmp_path):
     assert len(groups) == 1
     assert len(groups[0].files) == 2
+
+
+def test_empty_files_are_duplicates_of_each_other(tmp_path):
+    (tmp_path / "a.txt").write_text("")
+    (tmp_path / "b.txt").write_text("")
+    groups = find_duplicates(scan(tmp_path, Config()))
+    assert len(groups) == 1
"""

print(PRIMER_DIFF)

## Sortowanie diff linia po linijce

In [ ]:
stroki = PRIMER_DIFF.splitlines()

izmenennye_fajly = [s.split()[-1][2:] for s in stroki if s.startswith("diff --git")]
dobavlennye_stroki = [s for s in stroki if s.startswith("+") and not s.startswith("+++")]
udalennye_stroki = [s for s in stroki if s.startswith("-") and not s.startswith("---")]

print("Изменённые файлы:", izmenennye_fajly)
print("Добавлено строк:", len(dobavlennye_stroki))
print("Удалено строк:", len(udalennye_stroki))

## Weryfikacja

In [ ]:
assert izmenennye_fajly == ["src/safesort/duplicates.py", "tests/test_duplicates.py"]
assert len(dobavlennye_stroki) > 0
assert len(udalennye_stroki) == 0  # в этом diff ничего не удалено, только добавлено
print("Верно: diff затронул два файла, и в нём только добавления.")

## Starter

Wypełnij zaznaczone miejsce. Niezmieniony starter nie przechodzi tests.

In [ ]:
moe_commit_soobshenie = ""
# TODO: write a specific message with an accepted prefix.


## Task

Sformułuj logiczną commit message modyfikacji testu pliku null. Nie używaj `update` lub `fix stuff`.

## Tests

Run After task cell: jest podstawowy przykład i przynajmniej jeden skrajny przypadek.

In [ ]:
dopustimye_prefiksy = ("feat:", "fix:", "test:", "docs:", "refactor:", "chore:")
assert moe_commit_soobshenie.startswith(dopustimye_prefiksy)
assert len(moe_commit_soobshenie.split()) >= 4
assert moe_commit_soobshenie.lower() not in {"update", "fix", "fix stuff"}
print("Tests passed")

## Hint

Zgłaszaj nie samego faktu edycji, lecz badanej zmiany zachowania, na przykład, `test: ...`.

## Solution

Pokaż rozwiązanie po własnej próbie</summary>

```python
moe_commit_soobshenie = "feat: treat zero-byte files as duplicates of each other"

dopustimye_prefiksy = ("feat:", "fix:", "test:", "docs:", "refactor:", "chore:")

assert moe_commit_soobshenie.strip() != ""
assert moe_commit_soobshenie.startswith(dopustimye_prefiksy)
print("Верно:", moe_commit_soobshenie)
```

</details>